In [ ]:
import sys
import torch
import h5py
import openslide

import numpy as np
import matplotlib.pyplot as plt

sys.path.append('../')
from embeddings.embeddings import get_mixture_params
from utils.visualization_utils import get_panther_encoder, get_mixture_plot_figure, find_patch_size, find_fold



In [ ]:
def visualize_pt(slide_id, case_id, type, mode):
    """Visualize prototypes using patches of a single wsi."""
    # input paths
    slide_fpath = f'../data/data_files/tcga_{type}/wsi/images/{slide_id}.svs'
    h5_feats_fpath = f'../data/data_files/tcga_{type}/wsi/extracted_res0_5_patch256_uni/feats_h5/{slide_id}.h5'
    split_folder = find_fold(slide_id, type, mode)
    

    # Get WSI and feats
    wsi = openslide.open_slide(slide_fpath)
    h5 = h5py.File(h5_feats_fpath, 'r')
    feats = torch.Tensor(h5['features'][:]).unsqueeze(0)

    # Get PANTHER model and the wsi's to obtain the embeddings
    panther_encoder = get_panther_encoder(split_folder=split_folder)

    fig, axes = plt.subplots(16, 3, figsize=(1.5*3, 16 * 1.8))
    fig.suptitle(f"Case: {case_id}", fontsize=14, fontweight='bold', y=0.999)
    # Get proportions of each mixture component
    with torch.inference_mode():
        out, qqs = panther_encoder.representation(feats).values()
        pis, mus, vars = get_mixture_params(out, p=16)
        pis = pis[0].detach().cpu().numpy()
        qq = qqs[0,:,:,0].cpu().numpy()

        # Get the mixture plot
        display(get_mixture_plot_figure(pis))

        # Show closest patces for each prototype
        for pt in range(16):
            top_indices = np.argsort(qq[:, pt])[-3:][::-1]  # top 3, best first
            coords = h5['coords']
            
            for i, top_index in enumerate(top_indices):
                ax = axes[pt, i]
                if qq[top_index, pt] < 0.00001:
                    ax.imshow(np.ones((256, 256, 3)))  # blank white image
                else:
                    next_coords = coords[top_index+1:]
                    prev_coords = coords[:top_index]
                    coords_patch = coords[top_index]
                    patch_size = find_patch_size(next_coords, prev_coords, coords_patch)
                    patch = wsi.read_region(
                        (coords[top_index][0], coords[top_index][1]),
                        level=0,
                        size=(patch_size, patch_size)
                    ).convert("RGB")   
                    ax.imshow(patch)

                ax.axis("off")
                if i == 0:
                    if pis[pt] < 0.0005:
                        ax.text(-0.05, 0.5, f"$\\mathbf{{W({pt})}}$, c<0.001", va='center', ha='right', rotation=90, fontsize=9, transform=ax.transAxes)
                    else:
                        ax.text(-0.05, 0.5, f"$\\mathbf{{W({pt})}}$, c={pis[pt]:.3f}", va='center', ha='right', rotation=90, fontsize=9, transform=ax.transAxes)


    plt.tight_layout()
    plt.show()

def visualize_whole_wsi(file_path):
    """ Opens and visualizes the whole WSI """
    slide = openslide.OpenSlide(file_path)
    
    img_size = (2000, 2000) 
    image = slide.get_thumbnail(img_size)
    
    # Convert to numpy array for matplotlib
    image_rgb = np.array(image.convert("RGB"))
    
    # Display
    plt.figure(figsize=(10, 10))
    plt.imshow(image_rgb)
    plt.axis('off')
    plt.show()
    
    slide.close()


# Visualize the WSI features of a specific case

In [ ]:

slide_id = "TCGA-5T-A9QA-01Z-00-DX1.B4212117-E0A7-4EF2-B324-8396042ACEC1"
case_id = "-".join(slide_id.split('-')[:3])
type = 'brca'
    
visualize_pt(slide_id, case_id, type, mode="test")

# Visualize WSI

In [ ]:
slide_fpath = f'../data/data_files/tcga_{type}/wsi/images/{slide_id}.svs'
visualize_whole_wsi(slide_fpath)